# Deteção de anomalias nos CPE com base nas features extraídas

Este notebook identifica CPE com comportamento anómalo, com base nas features do Set A (perfil horário normalizado) e nos clusters previamente identificados.

Pretende-se:
- comparar múltiplos métodos de deteção de anomalias;
- comparar deteção global com deteção contextual (por cluster);
- analisar quais horas do dia mais distinguem CPEs anómalos;
- quantificar a concordância entre métodos;
- reutilizar o PCA persistido para consistência visual.

Nota: as features do Set A já estão na mesma escala (0-100%), pelo que NÃO é necessário aplicar StandardScaler.

In [ ]:
# Imports e configuração

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
 
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA
 
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")
 
base_dir = Path("../results")

# Pastas principais
exploration_dir = base_dir / "exploration"
features_dir = base_dir / "features"
clustering_dir = base_dir / "clustering"
anomalies_dir = base_dir / "anomalies"
models_dir = base_dir / "models"

# Subpastas específicas
intermediate_dir = exploration_dir / "intermediate"

# Clustering
clustering_analysis_dir = clustering_dir / "analysis"
clustering_plots_dir = clustering_dir / "plots"

# Features
features_plots_dir = features_dir / "plots"
features_validation_dir = features_dir / "validation"

# Anomalies
anomalies_plots_dir = anomalies_dir / "plots"
anomalies_temporal_plots_dir = anomalies_dir / "anomalies_temporal_plots"

for d in [
    exploration_dir,
    features_dir,
    clustering_dir,
    anomalies_dir,
    models_dir,
    intermediate_dir,
    clustering_analysis_dir,
    clustering_plots_dir,
    features_plots_dir,
    features_validation_dir,
    anomalies_plots_dir,
    anomalies_temporal_plots_dir,
]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# Carregamento dos dados e modelos persistidos

features = pd.read_csv(features_dir / "features_setA.csv", index_col=0)
clusters = pd.read_csv(clustering_dir / "clusters_cpe.csv", index_col=0)
 
print("Features:", features.shape)
print("Clusters:", clusters.shape)
 
# Carregar PCA do clustering (garante consistência visual)
pca = joblib.load(models_dir / "pca.pkl")
print(f"\nPCA carregado: {models_dir / 'pca.pkl'}")
 
# Juntar features com informação de cluster
df_anom = features.join(clusters[["cluster"]], how="inner")
 
print(f"\nDistribuição por cluster:")
display(df_anom["cluster"].value_counts())
 
# Separar outliers prévios (do clustering)
df_anom_clean = df_anom[df_anom["cluster"] != "outlier"].copy()
df_anom_outliers_previos = df_anom[df_anom["cluster"] == "outlier"].copy()
 
print(f"\nShape sem outliers prévios: {df_anom_clean.shape}")
print(f"Outliers prévios: {df_anom_outliers_previos.shape[0]}")

# Observações

O PCA é reutilizado do notebook de clustering para garantir consistência nas visualizações. As features do Set A (% por hora) NÃO requerem normalização — todos os métodos são aplicados sobre os dados originais.

In [ ]:
# Preparação dos dados

X = df_anom_clean.drop(columns=["cluster"])
 
# Sem StandardScaler — features já na mesma escala (%)
X_values = X.values
 
# PCA reutilizado do clustering
X_pca = pca.transform(X_values)
df_pca = pd.DataFrame(X_pca, index=X.index, columns=["PC1", "PC2"])
 
explained_variance = pca.explained_variance_ratio_
print(f"Variância explicada (PCA reutilizado):")
print(f"  PC1: {explained_variance[0]:.2%}")
print(f"  PC2: {explained_variance[1]:.2%}")

In [ ]:
# 3. Deteção GLOBAL de anomalias — comparação de métodos

print("DETEÇÃO GLOBAL DE ANOMALIAS")
 
CONTAMINATION = 0.05
 
metodos_global = {}
 
# Isolation Forest
iso = IsolationForest(contamination=CONTAMINATION, random_state=42)
labels_iso = iso.fit_predict(X_values)
scores_iso = iso.decision_function(X_values)
 
metodos_global["IsolationForest"] = {
    "labels": labels_iso,
    "scores": scores_iso,
    "n_anomalias": (labels_iso == -1).sum(),
}
 
# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=10, contamination=CONTAMINATION)
labels_lof = lof.fit_predict(X_values)
scores_lof = lof.negative_outlier_factor_
 
metodos_global["LOF"] = {
    "labels": labels_lof,
    "scores": scores_lof,
    "n_anomalias": (labels_lof == -1).sum(),
}
 
# Elliptic Envelope
try:
    ee = EllipticEnvelope(contamination=CONTAMINATION, random_state=42)
    labels_ee = ee.fit_predict(X_values)
    scores_ee = ee.decision_function(X_values)
    
    metodos_global["EllipticEnvelope"] = {
        "labels": labels_ee,
        "scores": scores_ee,
        "n_anomalias": (labels_ee == -1).sum(),
    }
except Exception as e:
    print(f"  EllipticEnvelope falhou (possível singularidade): {e}")
 
# Tabela comparativa global
print(f"\nContamination: {CONTAMINATION}")
for method, res in metodos_global.items():
    anomalos = X.index[res["labels"] == -1].tolist()
    print(f"\n  {method}: {res['n_anomalias']} anomalias")
    print(f"    CPEs: {anomalos}")

In [ ]:
# Concordância entre métodos (global)

df_concordancia = pd.DataFrame(index=X.index)
for method, res in metodos_global.items():
    df_concordancia[method] = np.where(res["labels"] == -1, 1, 0)
 
df_concordancia["n_metodos_anomalo"] = df_concordancia.sum(axis=1)
df_concordancia["anomalia_consenso"] = np.where(
    df_concordancia["n_metodos_anomalo"] >= 2, "anomalia", "normal"
)
 
print("CONCORDÂNCIA ENTRE MÉTODOS (GLOBAL)")
 
print(f"\nCPEs classificados como anómalos por número de métodos:")
display(df_concordancia["n_metodos_anomalo"].value_counts().sort_index())
 
cpes_anomalos_consenso = df_concordancia[df_concordancia["anomalia_consenso"] == "anomalia"].index.tolist()
print(f"\nCPEs anómalos por consenso (≥2 métodos): {len(cpes_anomalos_consenso)}")
for cpe in cpes_anomalos_consenso:
    metodos = [m for m in metodos_global if df_concordancia.loc[cpe, m] == 1]
    print(f"  {cpe} → detetado por: {', '.join(metodos)}")
 
# Matriz de concordância
method_names = list(metodos_global.keys())
concordancia_matrix = pd.DataFrame(index=method_names, columns=method_names, dtype=float)
for m1 in method_names:
    for m2 in method_names:
        concordancia_matrix.loc[m1, m2] = (df_concordancia[m1] == df_concordancia[m2]).mean()
 
print("\nMatriz de concordância (% de acordo):")
display(concordancia_matrix.round(3))
 
# Visualização PCA — todos os métodos
fig, axes = plt.subplots(1, len(metodos_global), figsize=(5*len(metodos_global), 5))
if len(metodos_global) == 1:
    axes = [axes]
 
for ax, (method, res) in zip(axes, metodos_global.items()):
    colors = ["red" if l == -1 else "blue" for l in res["labels"]]
    ax.scatter(df_pca["PC1"], df_pca["PC2"], c=colors, alpha=0.7, s=40)
    ax.set_title(f"{method}\n{res['n_anomalias']} anomalias")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.3)
 
plt.suptitle("Deteção global de anomalias — comparação de métodos", fontweight="bold")
plt.tight_layout()
plt.savefig(anomalies_plots_dir / "anomalias_globais_comparacao.png", dpi=150)
plt.show()

In [ ]:
# Deteção CONTEXTUAL de anomalias (por cluster)

print("DETEÇÃO CONTEXTUAL DE ANOMALIAS (POR CLUSTER)")
 
df_cluster_results = df_anom_clean.copy()
 
label_cols = []
for method_name in ["IsolationForest", "LOF", "EllipticEnvelope"]:
    df_cluster_results[f"{method_name}_label"] = "normal"
    df_cluster_results[f"{method_name}_score"] = np.nan
    label_cols.append(f"{method_name}_label")
 
for cluster_id in sorted(df_anom_clean["cluster"].unique()):
    
    df_c = df_anom_clean[df_anom_clean["cluster"] == cluster_id]
    X_c = df_c.drop(columns=["cluster"]).values
    
    n_samples = len(X_c)
    print(f"\nCluster {cluster_id} ({n_samples} CPEs)")
    
    # Isolation Forest — sem normalização (features já em %)
    try:
        iso_c = IsolationForest(contamination=CONTAMINATION, random_state=42)
        labels_c = iso_c.fit_predict(X_c)
        scores_c = iso_c.decision_function(X_c)
        
        df_cluster_results.loc[df_c.index, "IsolationForest_label"] = [
            "anomalia" if l == -1 else "normal" for l in labels_c
        ]
        df_cluster_results.loc[df_c.index, "IsolationForest_score"] = scores_c
        print(f"  IsolationForest: {(labels_c == -1).sum()} anomalias")
    except Exception as e:
        print(f"  IsolationForest falhou: {e}")
    
    # LOF
    try:
        n_neighbors = min(10, n_samples - 1)
        lof_c = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=CONTAMINATION)
        labels_c = lof_c.fit_predict(X_c)
        scores_c = lof_c.negative_outlier_factor_
        
        df_cluster_results.loc[df_c.index, "LOF_label"] = [
            "anomalia" if l == -1 else "normal" for l in labels_c
        ]
        df_cluster_results.loc[df_c.index, "LOF_score"] = scores_c
        print(f"  LOF: {(labels_c == -1).sum()} anomalias")
    except Exception as e:
        print(f"  LOF falhou: {e}")
    
    # Elliptic Envelope
    try:
        if n_samples > X_c.shape[1]:
            ee_c = EllipticEnvelope(contamination=CONTAMINATION, random_state=42)
            labels_c = ee_c.fit_predict(X_c)
            scores_c = ee_c.decision_function(X_c)
            
            df_cluster_results.loc[df_c.index, "EllipticEnvelope_label"] = [
                "anomalia" if l == -1 else "normal" for l in labels_c
            ]
            df_cluster_results.loc[df_c.index, "EllipticEnvelope_score"] = scores_c
            print(f"  EllipticEnvelope: {(labels_c == -1).sum()} anomalias")
        else:
            print(f"  EllipticEnvelope: cluster demasiado pequeno ({n_samples} < {X_c.shape[1]} features)")
    except Exception as e:
        print(f"  EllipticEnvelope falhou: {e}")

In [ ]:
# Concordância entre métodos (por cluster)

print("CONCORDÂNCIA ENTRE MÉTODOS (POR CLUSTER)")
 
df_cluster_results["n_metodos_anomalo_cluster"] = 0
for col in label_cols:
    df_cluster_results["n_metodos_anomalo_cluster"] += (df_cluster_results[col] == "anomalia").astype(int)
 
df_cluster_results["anomalia_consenso_cluster"] = np.where(
    df_cluster_results["n_metodos_anomalo_cluster"] >= 2, "anomalia", "normal"
)
 
print(f"\nDistribuição de concordância por cluster:")
display(
    df_cluster_results.groupby("cluster")["anomalia_consenso_cluster"]
    .value_counts()
    .unstack(fill_value=0)
)
 
cpes_anom_cluster = df_cluster_results[
    df_cluster_results["anomalia_consenso_cluster"] == "anomalia"
].index.tolist()
 
print(f"\nCPEs anómalos por consenso contextual (≥2 métodos): {len(cpes_anom_cluster)}")
for cpe in cpes_anom_cluster:
    cluster = df_cluster_results.loc[cpe, "cluster"]
    n_met = df_cluster_results.loc[cpe, "n_metodos_anomalo_cluster"]
    print(f"  {cpe} (cluster {cluster}) → {n_met} métodos concordam")

In [ ]:
# Comparação Global vs Contextual

print("COMPARAÇÃO: GLOBAL vs CONTEXTUAL")
 
df_compare = pd.DataFrame(index=X.index)
df_compare["cluster"] = df_anom_clean["cluster"]
df_compare["anomalia_global"] = df_concordancia["anomalia_consenso"]
df_compare["anomalia_cluster"] = df_cluster_results["anomalia_consenso_cluster"]
 
ct = pd.crosstab(
    df_compare["anomalia_global"],
    df_compare["anomalia_cluster"],
    margins=True
)
print("\nCrosstab Global vs Contextual (consenso ≥2 métodos):")
display(ct)
 
# Discordâncias
global_sim_cluster_nao = df_compare[
    (df_compare["anomalia_global"] == "anomalia") &
    (df_compare["anomalia_cluster"] == "normal")
]
global_nao_cluster_sim = df_compare[
    (df_compare["anomalia_global"] == "normal") &
    (df_compare["anomalia_cluster"] == "anomalia")
]
 
print(f"\nAnómalo global mas normal no cluster: {len(global_sim_cluster_nao)} CPEs")
if len(global_sim_cluster_nao) > 0:
    print("  → Estes CPEs são 'diferentes do global' mas normais dentro do seu grupo")
    display(global_sim_cluster_nao)
 
print(f"\nNormal global mas anómalo no cluster: {len(global_nao_cluster_sim)} CPEs")
if len(global_nao_cluster_sim) > 0:
    print("  → Estes CPEs parecem normais globalmente mas são atípicos no seu grupo")
    display(global_nao_cluster_sim)
 
# Visualização lado a lado
df_pca_plot = df_pca.copy()
df_pca_plot["cluster"] = df_anom_clean["cluster"]
df_pca_plot["anomalia_global"] = df_concordancia["anomalia_consenso"]
df_pca_plot["anomalia_cluster"] = df_cluster_results["anomalia_consenso_cluster"]
 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
 
colors_global = ["red" if a == "anomalia" else "blue" for a in df_pca_plot["anomalia_global"]]
axes[0].scatter(df_pca_plot["PC1"], df_pca_plot["PC2"], c=colors_global, alpha=0.7, s=40)
axes[0].set_title(f"Deteção Global (consenso)\n{(df_pca_plot['anomalia_global']=='anomalia').sum()} anomalias")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2"); axes[0].grid(True, alpha=0.3)
 
colors_cluster = ["red" if a == "anomalia" else "blue" for a in df_pca_plot["anomalia_cluster"]]
axes[1].scatter(df_pca_plot["PC1"], df_pca_plot["PC2"], c=colors_cluster, alpha=0.7, s=40)
axes[1].set_title(f"Deteção Contextual (consenso)\n{(df_pca_plot['anomalia_cluster']=='anomalia').sum()} anomalias")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2"); axes[1].grid(True, alpha=0.3)
 
plt.suptitle("Comparação: Deteção Global vs Contextual", fontweight="bold")
plt.tight_layout()
plt.savefig(anomalies_plots_dir / "anomalias_global_vs_cluster.png", dpi=150)
plt.show()
 
# Visualização combinada
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df_pca_plot,
    x="PC1", y="PC2",
    hue="cluster",
    style="anomalia_cluster",
    style_order=["normal", "anomalia"],
    markers={"normal": "o", "anomalia": "X"},
    palette="tab10",
    s=80, alpha=0.8
)
plt.title("Clusters e anomalias contextuais (projeção PCA)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(anomalies_plots_dir / "clusters_anomalias_pca.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Análise das horas que mais distinguem anomalias

print("HORAS QUE DISTINGUEM CPEs ANÓMALOS")
 
df_feat_anom = df_anom_clean.copy()
df_feat_anom["anomalia"] = df_cluster_results["anomalia_consenso_cluster"]
 
if (df_feat_anom["anomalia"] == "anomalia").sum() > 0:
    
    feat_cols = [c for c in df_feat_anom.columns if c not in ["cluster", "anomalia"]]
    
    mean_normal = df_feat_anom[df_feat_anom["anomalia"] == "normal"][feat_cols].mean()
    mean_anomalo = df_feat_anom[df_feat_anom["anomalia"] == "anomalia"][feat_cols].mean()
    
    # Effect size
    std_global = df_feat_anom[feat_cols].std()
    diff_norm = ((mean_anomalo - mean_normal) / std_global).abs().sort_values(ascending=False)
    
    print("\nHoras com maior diferença entre normais e anómalos:")
    print("(effect size = |mean_anom - mean_normal| / std_global)\n")
    display(diff_norm.head(10).round(3))
    
    # Perfil horário: normais vs anómalos
    horas = list(range(24))
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(horas, mean_normal.values, marker="o", markersize=4,
            label="Normais", linewidth=2, color="#3498DB")
    ax.plot(horas, mean_anomalo.values, marker="s", markersize=4,
            label="Anómalos", linewidth=2, color="#E74C3C")
    
    ax.set_title("Perfil horário médio: normais vs anómalos")
    ax.set_xlabel("Hora do dia")
    ax.set_ylabel("% do consumo diário")
    ax.set_xticks(horas)
    ax.set_xticklabels([f"{h}h" for h in horas], rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(anomalies_plots_dir / "perfil_normais_vs_anomalos.png", dpi=150)
    plt.show()
    
    # Boxplots das horas mais discriminativas
    top_features = diff_norm.head(8).index.tolist()
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    axes_flat = axes.flatten()
    
    for i, feat in enumerate(top_features):
        if i < len(axes_flat):
            hora_num = feat.split("_")[-1]
            sns.boxplot(
                data=df_feat_anom, x="anomalia", y=feat,
                ax=axes_flat[i], palette={"normal": "#3498DB", "anomalia": "#E74C3C"}
            )
            axes_flat[i].set_title(f"Hora {hora_num}\n(effect={diff_norm[feat]:.2f})", fontsize=9)
    
    for j in range(len(top_features), len(axes_flat)):
        axes_flat[j].set_visible(False)
    
    plt.suptitle("Horas mais discriminativas (normal vs anómalo)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(anomalies_plots_dir / "horas_discriminativas.png", dpi=150)
    plt.show()
 
else:
    print("Nenhuma anomalia detetada por consenso contextual.")

In [ ]:
# Análise de sensibilidade ao contamination

print("SENSIBILIDADE AO CONTAMINATION")
 
contam_range = [0.02, 0.05, 0.08, 0.10, 0.15]
sensibilidade = []
 
for contam in contam_range:
    iso_s = IsolationForest(contamination=contam, random_state=42)
    labels_s = iso_s.fit_predict(X_values)
    n_anom = (labels_s == -1).sum()
    
    sensibilidade.append({
        "contamination": contam,
        "n_anomalias": n_anom,
        "pct_anomalias": n_anom / len(labels_s) * 100,
        "cpes_anomalos": set(X.index[labels_s == -1].tolist())
    })
 
df_sens = pd.DataFrame(sensibilidade)[["contamination", "n_anomalias", "pct_anomalias"]]
display(df_sens)
 
# Estabilidade
todos_anom_sets = [s["cpes_anomalos"] for s in sensibilidade]
sempre_anomalo = set.intersection(*todos_anom_sets) if todos_anom_sets else set()
 
print(f"\nCPEs anómalos em TODOS os níveis de contamination testados:")
if sempre_anomalo:
    for cpe in sempre_anomalo:
        print(f"  {cpe}")
else:
    print("  Nenhum (os resultados variam com o contamination)")
 
# Frequência
freq_anomalo = {}
for s in sensibilidade:
    for cpe in s["cpes_anomalos"]:
        freq_anomalo[cpe] = freq_anomalo.get(cpe, 0) + 1
 
if freq_anomalo:
    df_freq = (
        pd.Series(freq_anomalo, name="n_vezes_anomalo")
        .sort_values(ascending=False)
        .to_frame()
    )
    df_freq["pct_configs"] = df_freq["n_vezes_anomalo"] / len(contam_range) * 100
    print(f"\nFrequência de classificação como anómalo (IsolationForest global):")
    display(df_freq)

In [ ]:
# Exportação de resultados

export_cluster = df_cluster_results[
    ["cluster"] + label_cols + ["n_metodos_anomalo_cluster", "anomalia_consenso_cluster"]
].copy()
export_cluster.to_csv(anomalies_dir / "anomalias_por_cluster.csv")
print("\nAnomalias por cluster guardadas:", anomalies_dir / "anomalias_por_cluster.csv")
 
export_global = df_concordancia.copy()
export_global.to_csv(anomalies_dir / "anomalias_globais.csv")
print("Anomalias globais guardadas:", anomalies_dir / "anomalias_globais.csv")
 
export_compare = df_compare.copy()
export_compare.to_csv(anomalies_dir / "anomalias_comparacao.csv")
print("Comparação guardada:", anomalies_dir / "anomalias_comparacao.csv")

# Conclusões

A deteção de anomalias com o Set A (perfil horário em %) permite identificar CPEs cujo padrão de distribuição horária do consumo se desvia do esperado para o seu grupo.

Vantagens do Set A para deteção de anomalias:
- A interpretação é directa: "este CPE consome X% mais à noite do que os seus pares no mesmo cluster"
- Sem necessidade de normalização (menos parâmetros, mais robusto)
- Consistente com o clustering realizado no mesmo espaço de features

Principais conclusões:
- O uso de consenso (≥2 métodos) reduz falsos positivos
- A abordagem contextual é mais precisa que a global
- As horas mais discriminativas revelam onde o CPE anómalo se desvia
- A sensibilidade ao contamination mostra a robustez dos resultados

Próximo passo: deteção de anomalias diretamente nas séries temporais